
# Chapter 6: Sequential Recommendation — SASRec and BERT4Rec

Companion notebook for **Chapter 6** of *Modern Recommender Systems*.

Change `MODEL` below to switch between the two architectures. Everything
else — data pipeline, training loop, evaluation, and ablations — is
identical for both. Results are tracked in MLFlow so you can compare the
two models side by side in the UI after running each.

**Requirements**
```
pip install torch pandas numpy requests mlflow
pip install -e .   # installs the recsys package from the repo root
```

**To compare SASRec vs BERT4Rec:**
1. Run this notebook with `MODEL = "sasrec"`
2. Change to `MODEL = "bert4rec"` and run again
3. Open the MLFlow UI: `mlflow ui` (from the repo root)
4. Both runs appear under the experiment `ch06_sequential` — use the
   comparison view to see loss curves, NDCG@10, and ablation results
   side by side.

**Hardware note:** training on the full MovieLens 25M dataset takes roughly
40 minutes per epoch on an A100. Set `QUICK_TEST = True` to verify
everything runs end-to-end in a few minutes before committing to a full run.


In [ ]:

# ── THE ONLY VARIABLE YOU NEED TO CHANGE ────────────────────────────────────
MODEL = "sasrec"   # "sasrec"  or  "bert4rec"
# ────────────────────────────────────────────────────────────────────────────

QUICK_TEST = True  # set False to reproduce the full Chapter 6 numbers

import os, zipfile, time
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import mlflow
import mlflow.pytorch

from recsys.fourstage_recsys.retrieval.sequential import (
    SASRec, BERT4Rec,
    gbce_loss, evaluate,
    SASRecTrainDataset, BERT4RecTrainDataset, SequentialEvalDataset,
    truncate_and_pad,
)

assert MODEL in ("sasrec", "bert4rec"), \
    f"MODEL must be 'sasrec' or 'bert4rec', got {MODEL!r}"

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Model: {MODEL.upper()}   Device: {device}")



## 1. MLFlow experiment setup

All runs for this notebook land in the `ch06_sequential` experiment.
Each invocation — one per `MODEL` value — is a separate run, named
`sasrec` or `bert4rec`, so the comparison view in the UI lines them
up correctly.


In [ ]:

MLFLOW_EXPERIMENT = "ch06_sequential"
GBCE_T    = 0.75
MASK_PROB = 0.2   # BERT4Rec only; logged as a param for both so runs are comparable

mlflow.set_experiment(MLFLOW_EXPERIMENT)
run = mlflow.start_run(run_name=MODEL)

mlflow.set_tags({
    "model":      MODEL,
    "quick_test": str(QUICK_TEST),
    "dataset":    "movielens-25m",
})
print(f"MLFlow run started: {run.info.run_id}")
print(f"Experiment: {MLFLOW_EXPERIMENT}   Run name: {MODEL}")
print("To view results: mlflow ui  (from the repo root)")



## 2. Data: MovieLens 25M


In [ ]:

DATA_DIR = "ml-25m"
URL = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"

def download_movielens():
    if os.path.exists(DATA_DIR):
        print(f"{DATA_DIR}/ already present, skipping download.")
        return
    import requests
    print("Downloading MovieLens 25M (~250 MB)...")
    with requests.get(URL, stream=True) as r:
        r.raise_for_status()
        with open("ml-25m.zip", "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                f.write(chunk)
    with zipfile.ZipFile("ml-25m.zip") as z:
        z.extractall(".")
    print("Done.")

download_movielens()


In [ ]:

ratings = pd.read_csv(os.path.join(DATA_DIR, "ratings.csv"))
ratings = ratings.sort_values(["userId", "timestamp"]).reset_index(drop=True)

if QUICK_TEST:
    sample_users = ratings["userId"].drop_duplicates().sample(n=3000, random_state=SEED)
    ratings = ratings[ratings["userId"].isin(sample_users)].reset_index(drop=True)

item_ids = ratings["movieId"].unique()
item_id_map = {old: new for new, old in enumerate(sorted(item_ids), start=1)}
ratings["item_idx"] = ratings["movieId"].map(item_id_map)
num_items = len(item_id_map)

user_sequences = (
    ratings.groupby("userId")["item_idx"]
    .apply(list)
    .reset_index(drop=True)
)
user_sequences = user_sequences[user_sequences.apply(len) >= 3].reset_index(drop=True)

mlflow.log_params({
    "num_users":    len(user_sequences),
    "num_items":    num_items,
    "quick_test":   QUICK_TEST,
    "dataset_size": len(ratings),
})
print(f"Interactions: {len(ratings):,}  Users: {len(user_sequences):,}  Items: {num_items:,}")



## 3. Train / validation / test split


In [ ]:

MAX_LEN = 200

train_seqs, val_inputs, val_targets, test_inputs, test_targets = [], [], [], [], []
for seq in user_sequences.tolist():
    train_seqs.append(seq[:-2])
    val_inputs.append(seq[:-2])
    val_targets.append(seq[-2])
    test_inputs.append(seq[:-1])
    test_targets.append(seq[-1])

val_dataset  = SequentialEvalDataset(val_inputs,  val_targets,  max_len=MAX_LEN)
test_dataset = SequentialEvalDataset(test_inputs, test_targets, max_len=MAX_LEN)
print(f"Train: {len(train_seqs):,}   Val: {len(val_targets):,}   Test: {len(test_targets):,}")



## 4. Model


In [ ]:

HIDDEN_DIM = 64
N_NEG      = 256 if not QUICK_TEST else 64
N_EPOCHS   = 20  if not QUICK_TEST else 2
PATIENCE   = 5

if MODEL == "sasrec":
    model = SASRec(
        num_items=num_items, max_len=MAX_LEN,
        hidden_dim=HIDDEN_DIM, num_layers=2, num_heads=2, dropout=0.2,
    )
    train_dataset = SASRecTrainDataset(
        train_seqs, num_items, max_len=MAX_LEN, n_neg=N_NEG,
    )
else:
    model = BERT4Rec(
        num_items=num_items, max_len=MAX_LEN,
        hidden_dim=HIDDEN_DIM, num_layers=2, num_heads=2, dropout=0.2,
        mask_prob=MASK_PROB,
    )
    train_dataset = BERT4RecTrainDataset(
        train_seqs, num_items, max_len=MAX_LEN, n_neg=N_NEG,
    )

model.to(device)
total_params = sum(p.numel() for p in model.parameters())

mlflow.log_params({
    "model":       MODEL,
    "hidden_dim":  HIDDEN_DIM,
    "num_layers":  2,
    "num_heads":   2,
    "dropout":     0.2,
    "max_len":     MAX_LEN,
    "n_neg":       N_NEG,
    "n_epochs":    N_EPOCHS,
    "patience":    PATIENCE,
    "gbce_t":      GBCE_T,
    "mask_prob":   MASK_PROB if MODEL == "bert4rec" else "n/a",
    "total_params": total_params,
})
print(f"{MODEL.upper()}  params={total_params:,}")



## 5. Training with gBCE and early stopping

Per-epoch training loss and validation NDCG@10 are logged to MLFlow so
you can plot the learning curves for both models in the UI.


In [ ]:

def train_one_epoch(model, loader, opt):
    model.train()
    total_loss, n_batches = 0.0, 0
    for batch in loader:
        if MODEL == "sasrec":
            input_seq, target_seq, neg_items = [b.to(device) for b in batch]
            pos_mask  = (target_seq != 0)
            hidden    = model(input_seq)
            pos_scores = (hidden * model.item_emb(target_seq)).sum(-1)
        else:
            raw_seq, neg_items = [b.to(device) for b in batch]
            input_seq, cloze_mask = model.mask_sequence(raw_seq)  # A
            pos_mask  = cloze_mask
            hidden    = model(input_seq)
            pos_scores = (hidden * model.item_emb(raw_seq)).sum(-1)  # B

        neg_emb    = model.item_emb(neg_items.to(device))
        neg_scores = (hidden.unsqueeze(2) * neg_emb).sum(-1)
        loss = gbce_loss(pos_scores, neg_scores, pos_mask, num_items, t=GBCE_T)

    # A  mask_sequence replaces some input tokens with [MASK] and returns a
    #    boolean mask indicating which positions were masked (the Cloze targets).
    #    This step is explicit in the training loop rather than hidden in the
    #    dataset so the data flow is visible: raw → masked → forward → loss.
    # B  The loss target is the original item at each masked position, so we
    #    score against raw_seq (not input_seq, which has [MASK] at those slots).

        opt.zero_grad()
        loss.backward()
        opt.step()
        total_loss += loss.item()
        n_batches  += 1
    return total_loss / n_batches


loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=0)
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

best_ndcg, best_state, epochs_no_improve = -1.0, None, 0

for epoch in range(N_EPOCHS):
    t0 = time.time()
    loss = train_one_epoch(model, loader, opt)
    val_metrics = evaluate(model, val_dataset, device=device)
    ndcg = val_metrics["NDCG@10"]
    elapsed = time.time() - t0

    mlflow.log_metrics({                              # A
        "train_loss":   loss,
        "val_ndcg10":   ndcg,
        "val_hr10":     val_metrics["HR@10"],
        "epoch_seconds": elapsed,
    }, step=epoch + 1)

    marker = ""
    if ndcg > best_ndcg:
        best_ndcg = ndcg
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        epochs_no_improve = 0
        marker = "  *"
    else:
        epochs_no_improve += 1

    print(f"Epoch {epoch+1:>2}/{N_EPOCHS}: loss={loss:.4f}  "
          f"val NDCG@10={ndcg:.4f}  ({elapsed:.1f}s){marker}")

    if epochs_no_improve >= PATIENCE:
        print(f"Early stopping after {PATIENCE} epochs without improvement.")
        break

model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
mlflow.log_metric("best_val_ndcg10", best_ndcg)      # B
print(f"\nRestored best checkpoint  (val NDCG@10={best_ndcg:.4f})")

# A Per-epoch metrics — these produce the learning-curve plots in the MLFlow UI.
# B The best validation score is also logged as a summary metric so it sorts
#   correctly in the experiment table without needing to open the run.



## 6. Test-set results


In [ ]:

test_metrics = evaluate(model, test_dataset, device=device)

mlflow.log_metrics({
    "test_ndcg10": test_metrics["NDCG@10"],
    "test_hr10":   test_metrics["HR@10"],
})

print(f"Test set — NDCG@10: {test_metrics['NDCG@10']:.4f}   HR@10: {test_metrics['HR@10']:.4f}")
print()
print("Note: full-catalog evaluation — the true item is ranked against")
print(f"all {num_items:,} items, not a sampled subset.")
print("Published benchmarks use sampled evaluation and are not directly comparable.")



## 7. Ablations


In [ ]:

@torch.no_grad()
def evaluate_ablation(model, eval_dataset, mode="full", seed=SEED):
    model.eval()
    loader = DataLoader(eval_dataset, batch_size=256, shuffle=False)
    rng = torch.Generator().manual_seed(seed)
    ndcgs, hits = [], []
    for seqs, targets in loader:
        seqs = seqs.clone()
        if mode == "shuffled":
            for i in range(seqs.shape[0]):
                valid = seqs[i] != 0
                n = int(valid.sum())
                if n > 1:
                    seqs[i, valid] = seqs[i, valid][torch.randperm(n, generator=rng)]
        elif mode == "last_only":
            last = seqs.gather(1, (seqs != 0).sum(1, keepdim=True) - 1)
            seqs = torch.zeros_like(seqs)
            seqs[:, -1] = last.squeeze(1)

        seqs, targets = seqs.to(device), targets.to(device)

        if MODEL == "bert4rec":
            mask_col = torch.full(
                (seqs.shape[0], 1), model.mask_token,
                dtype=torch.long, device=device,
            )
            seqs = torch.cat([seqs, mask_col], dim=1)[:, -model.max_len:]

        hidden = model(seqs)
        user_state = hidden[:, -1, :]
        scores = user_state @ model.item_emb.weight[:model.num_items + 1].T
        scores[:, 0] = -float("inf")

        topk = torch.topk(scores, 10, dim=1).indices
        hit  = (topk == targets.unsqueeze(1))
        rank = hit.float().argmax(dim=1)
        has_hit = hit.any(dim=1)
        ndcg = torch.where(
            has_hit,
            1.0 / torch.log2(rank.float() + 2),
            torch.zeros_like(rank, dtype=torch.float),
        )
        ndcgs.extend(ndcg.cpu().tolist())
        hits.extend(has_hit.float().cpu().tolist())
    return {"NDCG@10": float(np.mean(ndcgs)), "HR@10": float(np.mean(hits))}


In [ ]:

# Ablation A: retrain without positional embeddings.
class NoPosEmb(SASRec if MODEL == "sasrec" else BERT4Rec):
    def _encode(self, sequences, attn_mask=None):
        x = self.item_emb(sequences)
        x = self.input_dropout(x)
        padding_mask = (sequences == 0)
        for layer in self.transformer.layers:
            x = layer(x, src_mask=attn_mask, src_key_padding_mask=padding_mask)
            x = x.masked_fill(padding_mask.unsqueeze(-1), 0.0)
        return self.final_norm(x)

torch.manual_seed(SEED)
kwargs = dict(num_items=num_items, max_len=MAX_LEN, hidden_dim=HIDDEN_DIM,
              num_layers=2, num_heads=2, dropout=0.2)
if MODEL == "bert4rec":
    kwargs["mask_prob"] = MASK_PROB
model_no_pos = NoPosEmb(**kwargs).to(device)
opt_a = torch.optim.Adam(model_no_pos.parameters(), lr=1e-3)

print("Training Ablation A (no positional embeddings)...")
best_a, best_state_a = -1.0, None
for epoch in range(N_EPOCHS):
    loss_a = train_one_epoch(model_no_pos, loader, opt_a)
    m_a = evaluate(model_no_pos, val_dataset, device=device)
    if m_a["NDCG@10"] > best_a:
        best_a = m_a["NDCG@10"]
        best_state_a = {k: v.cpu().clone() for k, v in model_no_pos.state_dict().items()}
model_no_pos.load_state_dict({k: v.to(device) for k, v in best_state_a.items()})
print(f"  best val NDCG@10={best_a:.4f}")


In [ ]:

ablation_results = {
    "full":      evaluate_ablation(model,        test_dataset, mode="full"),
    "no_pos":    evaluate_ablation(model_no_pos, test_dataset, mode="full"),
    "shuffled":  evaluate_ablation(model,        test_dataset, mode="shuffled"),
    "last_only": evaluate_ablation(model,        test_dataset, mode="last_only"),
}

# Log each ablation result as a flat metric so the MLFlow comparison table
# shows all four rows for each model without needing to open the run.
for name, m in ablation_results.items():
    mlflow.log_metrics({
        f"ablation_{name}_ndcg10": m["NDCG@10"],
        f"ablation_{name}_hr10":   m["HR@10"],
    })

label = {
    "full":      "Full model",
    "no_pos":    "Ablation A: no pos embeddings",
    "shuffled":  "Ablation B: shuffled history",
    "last_only": "Ablation C: last item only",
}
print(f"{'Setup':<40}{'NDCG@10':>10}{'HR@10':>10}")
print("-" * 60)
for key, m in ablation_results.items():
    print(f"{label[key]:<40}{m['NDCG@10']:>10.4f}{m['HR@10']:>10.4f}")



## 8. Save model artifact and close the MLFlow run


In [ ]:

mlflow.pytorch.log_model(model, artifact_path="model")   # A
mlflow.end_run()

print(f"\nMLFlow run complete.")
print(f"  Experiment : {MLFLOW_EXPERIMENT}")
print(f"  Run name   : {MODEL}")
print(f"  Run ID     : {run.info.run_id}")
print()
print("To compare SASRec and BERT4Rec:")
print("  1. Change MODEL to 'bert4rec' and re-run the notebook")
print("  2. Run: mlflow ui")
print("  3. Select both runs in the experiment table → Compare")

# A  The trained model weights are stored as an MLFlow artifact, making it
#    possible to load either checkpoint later without re-training:
#    model = mlflow.pytorch.load_model(f"runs:/{run_id}/model")



## 9. Reading the results in MLFlow

Open the MLFlow UI (`mlflow ui` from the repo root, then visit
`http://localhost:5000`). Under the `ch06_sequential` experiment you will
see one run per model. Select both and click **Compare** to get:

- **Parallel coordinates plot** — how `hidden_dim`, `n_neg`, `gbce_t`, and
  `mask_prob` relate to `test_ndcg10` across the two runs.
- **Metric history charts** — `train_loss` and `val_ndcg10` plotted per
  epoch for both models on the same axes, so you can see whether one
  converges faster or to a higher value.
- **Table view** — all ablation metrics (`ablation_full_ndcg10`,
  `ablation_shuffled_ndcg10`, etc.) side by side.

The comparison the original BERT4Rec paper failed to make — both models
trained with the same gBCE loss on the same data — is now one click away.
